In [2]:

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold,GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, average_precision_score
from cdc_ml.config import POLLS_PROCESSED


2026-05-14 12:47:29.103 | INFO     | cdc_ml.config:<module>:12 - PROJ_ROOT path is: C:\Users\zhiju\Desktop\cdc_ml


In [8]:
test = np.arange(9)
print(test[[1,2,3]])

[1 2 3]


In [4]:
df = pd.read_parquet(POLLS_PROCESSED)

In [5]:

def add_features(df):
    df = df.copy()
    df["hours_into_cycle"] = (
        (df["polling_at"] - df["cycle_start"]).dt.total_seconds() / 3600.0
    )
    for col, period in [("polling_hour", 24), ("polling_dow", 7),
                        ("cycle_start_hour", 24), ("cycle_start_dow", 7)]:
        df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
        df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)
    return df

FEATURES = [
    "hours_into_cycle",
    "polling_hour_sin", "polling_hour_cos",
    "polling_dow_sin", "polling_dow_cos",
    "cycle_start_hour_sin", "cycle_start_hour_cos",
    "cycle_start_dow_sin", "cycle_start_dow_cos",
]

df = add_features(df)
df["cycle_id"] = df["cycle_start"].astype("int64")
y = df["has_booking"].to_numpy()
X = df[FEATURES].to_numpy()
groups = df["cycle_id"].to_numpy()

def score(y_true, p):
    return {
        "brier":     brier_score_loss(y_true, p),
        "pr_auc":    average_precision_score(y_true, p),
        "base_rate": y_true.mean(),
        "pred_mean": p.mean(),
    }

rows = []
for fold, (tr, va) in enumerate(GroupKFold(n_splits=5).split(X, y, groups)):
    print(fold)
    base = y[tr].mean()
    p_const = np.full(len(va), base)

    # joint marginal on (polling_dow, polling_hour) — pandas-native
    lut = (
        df.iloc[tr]
        .assign(_y=y[tr])
        .groupby(["polling_dow", "polling_hour"])["_y"]
        .mean()
    )
    idx = pd.MultiIndex.from_frame(df.iloc[va][["polling_dow", "polling_hour"]])
    p_marg = lut.reindex(idx).fillna(base).to_numpy()

    lr = LogisticRegression(max_iter=1000).fit(X[tr], y[tr])
    p_lr = lr.predict_proba(X[va])[:,1]

    rows += [
        {"fold": fold, "model": "const",   **score(y[va], p_const)},
        {"fold": fold, "model": "marg_dh", **score(y[va], p_marg)},
        {"fold": fold, "model": "logreg",  **score(y[va], p_lr)},
    ]

print(pd.DataFrame(rows).groupby("model").mean(numeric_only=True))

0
1
2
3
4
         fold     brier    pr_auc  base_rate  pred_mean
model                                                  
const     2.0  0.012041  0.012189   0.012189   0.012188
logreg    2.0  0.012002  0.025312   0.012189   0.012208
marg_dh   2.0  0.011984  0.027172   0.012189   0.012185


In [6]:

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    brier_score_loss, average_precision_score, log_loss,
)
import xgboost as xgb

# --- assume df already has hours_into_cycle ---
df["cycle_id"] = df.groupby(["username", "cycle_start"]).ngroup()

FEATURES = [
    "hours_into_cycle",
    "polling_hour", "polling_dow",
    "cycle_start_hour", "cycle_start_dow",
]

X = df[FEATURES]
y = df["has_booking"].to_numpy()
groups = df["cycle_id"].to_numpy()

def top_k_recall_per_cycle(y_true, p, cycle_ids, k):
    """Per cycle: take top-k cells by prob, return mean recall over cycles with ≥1 positive."""
    d = pd.DataFrame({"y": y_true, "p": p, "c": cycle_ids})
    recalls = []
    for _, g in d.groupby("c"):
        pos = g["y"].sum()
        if pos == 0:
            continue
        topk = g.nlargest(min(k, len(g)), "p")
        recalls.append(topk["y"].sum() / pos)
    return float(np.mean(recalls)) if recalls else np.nan

def score(y, p, cycle_ids):
    return {
        "brier":     brier_score_loss(y, p),
        "logloss":   log_loss(y, np.clip(p, 1e-7, 1 - 1e-7)),
        "pr_auc":    average_precision_score(y, p),
        "base_rate": y.mean(),
        "pred_mean": p.mean(),
        "recall@5":  top_k_recall_per_cycle(y, p, cycle_ids, 5),
        "recall@10": top_k_recall_per_cycle(y, p, cycle_ids, 10),
    }

params = dict(
    objective="binary:logistic",
    eval_metric=["logloss", "aucpr"],
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",
    n_jobs=-1,
)

rows, oof_p = [], np.zeros(len(y))
for fold, (tr, va) in enumerate(GroupKFold(n_splits=5).split(X, y, groups)):
    model = xgb.XGBClassifier(
        n_estimators=2000,
        early_stopping_rounds=100,
        **params,
    )
    model.fit(X.iloc[tr], y[tr], eval_set=[(X.iloc[va], y[va])], verbose=False)

    p = model.predict_proba(X.iloc[va])[:, 1]
    oof_p[va] = p

    m = score(y[va], p, groups[va])
    m["fold"] = fold
    m["best_iter"] = model.best_iteration
    rows.append(m)

cv = pd.DataFrame(rows)
print(cv)
print("\nMean across folds:")
print(cv.drop(columns=["fold", "best_iter"]).mean())

print(f"\nOOF Brier:    {brier_score_loss(y, oof_p):.5f}")
print(f"OOF PR-AUC:   {average_precision_score(y, oof_p):.5f}")
print(f"OOF recall@5: {top_k_recall_per_cycle(y, oof_p, groups, 5):.4f}")

      brier   logloss    pr_auc  base_rate  pred_mean  recall@5  recall@10  \
0  0.013292  0.065717  0.056285   0.013711   0.012805  0.120519   0.163870   
1  0.013952  0.070729  0.060931   0.014328   0.010901  0.244287   0.266091   
2  0.007947  0.046861  0.055620   0.008011   0.012783  0.111111   0.146825   
3  0.016200  0.081148  0.052593   0.016587   0.011365  0.128968   0.167989   
4  0.008192  0.046685  0.034777   0.008297   0.013974  0.111111   0.190476   

   fold  best_iter  
0     0         82  
1     1         58  
2     2          4  
3     3         14  
4     4         19  

Mean across folds:
brier        0.011916
logloss      0.062228
pr_auc       0.052041
base_rate    0.012187
pred_mean    0.012365
recall@5     0.143199
recall@10    0.187050
dtype: float64

OOF Brier:    0.01192
OOF PR-AUC:   0.03840
OOF recall@5: 0.1444
